In [1]:
# -*- coding: utf-8 -*-
"""nlp.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1ydRgvJer0-uJDneLDN-lYQqneomBxLBZ
"""

# -*- coding: utf-8 -*-
"""NLP AISAP Chatbot """

import os
import pandas as pd
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

# --- CONFIGURATION ---
DATA_PATH = "aisap_dataset.csv"
MODEL_NAME = "google/flan-t5-base"
OUTPUT_DIR = "flan-t5-ai-simple-bot"

def main():
    print(f"Loading dataset from {DATA_PATH}...")

    # 1. Load Dataset
    data_files = {"train": DATA_PATH}
    # Check if file exists first to avoid confusing errors
    if not os.path.exists(DATA_PATH):
        print(f"ERROR: File {DATA_PATH} not found. Please run the 'dataset generation' script first.")
        return

    raw_datasets = load_dataset("csv", data_files=data_files)
    raw_datasets = raw_datasets["train"].train_test_split(test_size=0.1, seed=42)

    # 2. Setup Model & Tokenizer
    print(f"Loading Model: {MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

    max_input_length = 256
    max_target_length = 256

    def preprocess_function(examples):
        # Instruction Tuning Prefix
        prefix = "Explain this AI concept with an analogy: "

        inputs = [prefix + str(doc) for doc in examples["category"]]
        targets = [str(doc) for doc in examples["simple_explanation"]]

        # Tokenize inputs
        model_inputs = tokenizer(
            inputs,
            max_length=max_input_length,
            truncation=True
        )

        # Tokenize targets (Updated to fix the warning you saw)
        labels = tokenizer(
            text_target=targets,
            max_length=max_target_length,
            truncation=True
        )

        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    print("Tokenizing data...")
    tokenized_datasets = raw_datasets.map(
        preprocess_function,
        batched=True,
        remove_columns=raw_datasets["train"].column_names,
    )

    # 3. Training Setup
    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

    training_args = Seq2SeqTrainingArguments(
        output_dir=OUTPUT_DIR,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=3e-4,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=1,
        weight_decay=0.01,
        save_total_limit=2,
        num_train_epochs=15,
        predict_with_generate=True,
        fp16=False,
        logging_steps=50,
        push_to_hub=False,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"],
        tokenizer=tokenizer,
        data_collator=data_collator,
    )

    # 4. Train
    print("Starting training...")
    trainer.train()

    # 5. Save
    print("Saving model...")
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)

    # 6. Start Chat
    chat(OUTPUT_DIR)

def chat(model_dir):
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    # Reload the trained model
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)

    print("\n" + "="*40)
    print("AISAP Bot Ready! (Type 'quit' to exit)")
    print("="*40)

    while True:
        user_input = input("\nYou: ")
        if user_input.lower() in ["quit", "exit"]:
            break

        # Must match the training prefix
        prompt = f"Explain this AI concept with an analogy: {user_input}"

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=256
        )

        output_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=5,
            temperature=0.7,
            early_stopping=True
        )

        answer = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        print(f"Bot: {answer}")

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'datasets'

In [ ]:
!pip install git+https://github.com/feralvam/easse.git
!pip install textstat bert-score


  Cloning https://github.com/feralvam/easse.git to /tmp/pip-req-build-o2cn1328
  Running command git clone --filter=blob:none --quiet https://github.com/feralvam/easse.git /tmp/pip-req-build-o2cn1328
  Resolved https://github.com/feralvam/easse.git to commit 6a4352ec299ed03fda8ee45445ca43d9c7673e89
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/facebookresearch/text-simplification-evaluation.git (to revision main) to /tmp/pip-install-vp540_ki/tseval_603c4a585e604722a235ea6cb634a572
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/text-simplification-evaluation.git /tmp/pip-install-vp540_ki/tseval_603c4a585e604722a235ea6cb634a572
  Resolved https://github.com/facebookresearch/text-simplification-evaluation.git to commit dea8863683ea5946fd50184883c9be7a7339e821
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   

In [ ]:
import json
import numpy as np
import pandas as pd
from typing import List, Dict
from easse.sari import corpus_sari
import textstat
from bert_score import score as bert_score
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset

class TextEvaluationMetrics:
    """
    A comprehensive evaluation class for text simplification metrics.
    Matches the FLAN-T5-base training configuration.
    """
    def __init__(self):
        self.results = {}

    def load_data(self, file_path: str) -> List[Dict]:

        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        return data

    def compute_sari(self, predictions: List[str], references: List[str],
                     sources: List[str] = None) -> float:
        """
        Compute SARI score for text simplification.
        Args:
            predictions: List of model outputs
            references: List of reference simplifications
            sources: List of source texts (optional, uses placeholder if None)
        Returns:
            SARI score (float)
        """
        if sources is None:
            sources = ["placeholder"] * len(predictions)

        sari_score = corpus_sari(
            orig_sents=sources,
            sys_sents=predictions,
            refs_sents=[references]  # Single set of references
        )
        return sari_score

    def compute_readability_metrics(self, texts: List[str]) -> Dict[str, float]:
        """
        Compute average readability metrics across all texts.
        Args:
            texts: List of text strings to analyze
        Returns:
            Dictionary with average readability metrics
        """
        fk_grades = []
        avg_sentence_lengths = []
        avg_syllables = []

        for text in texts:
            if text and text.strip():
                try:
                    fk_grades.append(textstat.flesch_kincaid_grade(text))
                    avg_sentence_lengths.append(textstat.avg_sentence_length(text))
                    avg_syllables.append(textstat.avg_syllables_per_word(text))
                except:
                    continue

        return {
            'flesch_kincaid_grade': np.mean(fk_grades) if fk_grades else 0.0,
            'avg_sentence_length': np.mean(avg_sentence_lengths) if avg_sentence_lengths else 0.0,
            'avg_syllables_per_word': np.mean(avg_syllables) if avg_syllables else 0.0
        }

    def compute_bertscore(self, candidates: List[str], references: List[str]) -> Dict[str, float]:
        """
        Compute BERTScore for semantic similarity.
        Args:
            candidates: List of model outputs
            references: List of reference texts
        Returns:
            Dictionary with precision, recall, and F1 scores
        """
        P, R, F1 = bert_score(
            cands=candidates,
            refs=references,
            lang="en",
            verbose=True
        )

        return {
            'precision': P.mean().item(),
            'recall': R.mean().item(),
            'f1': F1.mean().item()
        }

    def evaluate_model(self, predictions: List[str], references: List[str],
                      sources: List[str] = None, model_name: str = "Model") -> Dict:
        """
        Run all evaluation metrics for a model.
        Args:
            predictions: Model outputs
            references: Reference texts
            sources: Source texts (optional)
            model_name: Name identifier for the model
        Returns:
            Dictionary with all metrics
        """
        print(f"\n{'='*60}")
        print(f"Evaluating {model_name}")
        print(f"{'='*60}")

        # SARI Score
        print("\nComputing SARI score...")
        sari = self.compute_sari(predictions, references, sources)
        print(f"SARI Score: {sari:.4f}")

        # Readability Metrics
        print("\nComputing readability metrics...")
        readability = self.compute_readability_metrics(predictions)
        print(f"Flesch-Kincaid Grade: {readability['flesch_kincaid_grade']:.2f}")
        print(f"Avg Sentence Length: {readability['avg_sentence_length']:.2f}")
        print(f"Avg Syllables/Word: {readability['avg_syllables_per_word']:.2f}")

        # BERTScore
        print("\nComputing BERTScore...")
        bertscore = self.compute_bertscore(predictions, references)
        print(f"BERTScore Precision: {bertscore['precision']:.4f}")
        print(f"BERTScore Recall: {bertscore['recall']:.4f}")
        print(f"BERTScore F1: {bertscore['f1']:.4f}")

        # Compile all results
        results = {
            'model_name': model_name,
            'sari': sari,
            'readability': readability,
            'bertscore': bertscore
        }

        self.results[model_name] = results
        return results

    def save_results(self, output_path: str):
        """
        Save evaluation results to JSON file.
        Args:
            output_path: Path to save results
        """
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(self.results, f, indent=2)
        print(f"\nResults saved to {output_path}")


def generate_predictions_from_model(model_dir: str, test_inputs: List[str],
                                   max_new_tokens: int = 128) -> List[str]:
    """
    Generate predictions using your trained FLAN-T5-base model.
    Matches the exact configuration from your training script.

    Args:
        model_dir: Directory containing your trained model
        test_inputs: List of input terms/categories
        max_new_tokens: Maximum tokens to generate
    Returns:
        List of generated explanations
    """
    print(f"\nLoading model from {model_dir}...")
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)

    predictions = []
    for i, input_text in enumerate(test_inputs):
        # Must match the training prefix exactly
        prompt = f"Explain this AI concept with an analogy: {input_text}"

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=256
        )

        # Use same generation parameters as in your chat function
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=5,
            temperature=0.7,
            early_stopping=True
        )

        answer = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        predictions.append(answer.strip())

        # Progress indicator
        if (i + 1) % 10 == 0:
            print(f"Generated {i + 1}/{len(test_inputs)} predictions...")

    return predictions


# Example usage matching your training setup
if __name__ == "__main__":
    # --- CONFIGURATION (Match your training script) ---
    DATA_PATH = "aisap_dataset.csv"
    MODEL_DIR = "flan-t5-ai-simple-bot"
    RESULTS_PATH = "evaluation_results.json"

    # Initialize evaluator
    evaluator = TextEvaluationMetrics()

    # Load your test data from CSV
    print(f"Loading test data from {DATA_PATH}...")
    test_df = pd.read_csv(DATA_PATH)

    # Use the test split (10% that was held out during training)
    # Or take a specific subset for evaluation
    test_size = min(100, len(test_df))  # Evaluate on first 100 or all if less
    test_inputs = test_df["category"].head(test_size).tolist()
    test_references = test_df["simple_explanation"].head(test_size).tolist()
    test_sources = test_df["category"].head(test_size).tolist()

    print(f"Evaluating on {test_size} examples...")

    # Generate predictions from your trained model
    predictions = generate_predictions_from_model(MODEL_DIR, test_inputs)

    # Evaluate
    results = evaluator.evaluate_model(
        predictions=predictions,
        references=test_references,
        sources=test_sources,
        model_name="FLAN-T5-Base-AISAP"
    )

    # Save results
    evaluator.save_results(RESULTS_PATH)

    print("\n" + "="*60)
    print("Evaluation Complete!")
    print("="*60)
    print(f"\nSummary:")
    print(f"  - SARI Score: {results['sari']:.4f}")
    print(f"  - BERTScore F1: {results['bertscore']['f1']:.4f}")
    print(f"  - Flesch-Kincaid Grade: {results['readability']['flesch_kincaid_grade']:.2f}")
    print(f"\nDetailed results saved to: {RESULTS_PATH}")

Loading test data from aisap_dataset.csv...
Evaluating on 100 examples...

Loading model from flan-t5-ai-simple-bot...
Generated 10/100 predictions...
Generated 20/100 predictions...
Generated 30/100 predictions...
Generated 40/100 predictions...
Generated 50/100 predictions...
Generated 60/100 predictions...
Generated 70/100 predictions...
Generated 80/100 predictions...
Generated 90/100 predictions...
Generated 100/100 predictions...

Evaluating FLAN-T5-Base-AISAP

Computing SARI score...
SARI Score: 66.2941

Computing readability metrics...


/tmp/ipython-input-225357495.py:62: DeprecationWarning: The 'avg_sentence_length' method has been deprecated due to being the same as 'words_per_sentence'. This method will be removed in thefuture.
  avg_sentence_lengths.append(textstat.avg_sentence_length(text))


Flesch-Kincaid Grade: 13.70
Avg Sentence Length: 18.57
Avg Syllables/Word: 1.87

Computing BERTScore...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/3 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 1.72 seconds, 58.12 sentences/sec
BERTScore Precision: 0.9507
BERTScore Recall: 0.9453
BERTScore F1: 0.9479

Results saved to evaluation_results.json

Evaluation Complete!

Summary:
  - SARI Score: 66.2941
  - BERTScore F1: 0.9479
  - Flesch-Kincaid Grade: 13.70

Detailed results saved to: evaluation_results.json
